# Ventana Computacional IX.1  
## Koopman/EDMD para el oscilador de Duffing forzado

Este notebook acompaña el **Capítulo IX** de la monografía.  
El objetivo es mostrar, paso a paso, cómo una dinámica no lineal puede estudiarse mediante observables y una aproximación finita del operador de Koopman.

Trabajaremos con el oscilador de Duffing forzado:

$$
\ddot{x}+\delta \dot{x}+\alpha x+\beta x^3=\gamma\cos(\Omega t).
$$

Para tratarlo como sistema autónomo, introducimos una fase externa:

$$
\theta = \Omega t,\qquad \dot{\theta}=\Omega.
$$

Entonces:

$$
\dot{x}=v,
$$

$$
\dot{v}=-\delta v-\alpha x-\beta x^3+\gamma\cos\theta,
$$

$$
\dot{\theta}=\Omega.
$$

## 1. Importar librerías

Usaremos `numpy`, `scipy` y `matplotlib`.  
En Colab normalmente estas librerías ya están instaladas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.linalg import pinv

np.set_printoptions(precision=4, suppress=True)

## 2. Definir el sistema de Duffing forzado

Elegimos parámetros suaves para obtener una respuesta periódica estable, sin entrar todavía en regímenes caóticos.

$$
\delta=0.2,\qquad
\alpha=1,\qquad
\beta=0.1,\qquad
\gamma=0.3,\qquad
\Omega=1.
$$

In [ ]:
delta = 0.2
alpha = 1.0
beta = 0.1
gamma = 0.3
Omega = 1.0

def duffing_forzado(t, z):
    x, v, theta = z

    dx = v
    dv = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(theta)
    dtheta = Omega

    return [dx, dv, dtheta]

## 3. Simulación numérica

Simulamos durante un tiempo largo y después eliminamos el transitorio inicial.  
Esto nos permite estudiar la respuesta estacionaria.

In [ ]:
dt = 0.02
t0 = 0.0
tf = 200.0

t_eval = np.arange(t0, tf, dt)
z0 = [0.2, 0.0, 0.0]

sol = solve_ivp(
    duffing_forzado,
    (t0, tf),
    z0,
    t_eval=t_eval,
    rtol=1e-9,
    atol=1e-9
)

x_all = sol.y[0]
v_all = sol.y[1]
theta_all = np.mod(sol.y[2], 2*np.pi)
t_all = sol.t

print("Número de muestras:", len(t_all))
print("Estado final aproximado:", sol.y[:, -1])

## 4. Visualizar la trayectoria completa

Primero vemos la señal completa, incluyendo el transitorio.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t_all, x_all)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Duffing forzado: señal completa con transitorio")
plt.grid(True)
plt.show()

## 5. Eliminar transitorio

Quitamos los primeros 80 segundos de simulación.

In [ ]:
cut = int(80.0 / dt)

x = x_all[cut:]
v = v_all[cut:]
theta = theta_all[cut:]
t = t_all[cut:]

plt.figure(figsize=(10, 4))
plt.plot(t, x)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Respuesta estacionaria aproximada")
plt.grid(True)
plt.show()

## 6. Espacio de fase

La respuesta estacionaria puede verse en el plano $(x,v)$.

In [ ]:
plt.figure(figsize=(5, 5))
plt.plot(x, v)
plt.xlabel("x")
plt.ylabel("v")
plt.title("Plano de fase: respuesta estacionaria")
plt.grid(True)
plt.axis("equal")
plt.show()

## 7. Diccionario de observables

La idea de Koopman es no observar solamente el estado

$$
(x,v,\theta),
$$

sino funciones del estado:

$$
g(x,v,\theta).
$$

Usaremos el diccionario:

$$
\Psi =
\begin{bmatrix}
1,
x,
v,
x^2,
xv,
v^2,
x^3,
x^2v,
xv^2,
v^3,
\cos\theta,
\sin\theta,
x\cos\theta,
x\sin\theta,
v\cos\theta,
v\sin\theta
\end{bmatrix}^T.
$$

Este diccionario no es único. Elegir observables es una decisión de modelado.

In [ ]:
observable_names = [
    "1",
    "x",
    "v",
    "x^2",
    "xv",
    "v^2",
    "x^3",
    "x^2v",
    "xv^2",
    "v^3",
    "cos(theta)",
    "sin(theta)",
    "x cos(theta)",
    "x sin(theta)",
    "v cos(theta)",
    "v sin(theta)"
]

def psi(x, v, theta):
    return np.vstack([
        np.ones_like(x),
        x,
        v,
        x**2,
        x*v,
        v**2,
        x**3,
        x**2*v,
        x*v**2,
        v**3,
        np.cos(theta),
        np.sin(theta),
        x*np.cos(theta),
        x*np.sin(theta),
        v*np.cos(theta),
        v*np.sin(theta)
    ])

Psi = psi(x, v, theta)

print("Forma de Psi:", Psi.shape)
print("Número de observables:", Psi.shape[0])

## 8. Construcción de matrices para EDMD

Definimos:

$$
X =
\begin{bmatrix}
\Psi(z_0) & \Psi(z_1) & \cdots & \Psi(z_{N-1})
\end{bmatrix},
$$

$$
Y =
\begin{bmatrix}
\Psi(z_1) & \Psi(z_2) & \cdots & \Psi(z_N)
\end{bmatrix}.
$$

Buscamos una matriz discreta $K$ tal que:

$$
Y \approx KX.
$$

La solución de mínimos cuadrados es:

$$
K = YX^\dagger.
$$

In [ ]:
X = Psi[:, :-1]
Y = Psi[:, 1:]

K = Y @ pinv(X)

print("Forma de X:", X.shape)
print("Forma de Y:", Y.shape)
print("Forma de K:", K.shape)

## 9. Predicción iterada usando la matriz de Koopman aproximada

Usamos la matriz $K$ para propagar los observables:

$$
\Psi_{k+1} \approx K\Psi_k.
$$

Después recuperamos $x(t)$ porque el observable $x$ es la segunda entrada del diccionario.

In [ ]:
n_steps = X.shape[1]

psi_pred = np.zeros_like(X)
psi_pred[:, 0] = X[:, 0]

for k in range(n_steps - 1):
    psi_pred[:, k+1] = K @ psi_pred[:, k]

x_true = X[1, :]
x_pred = psi_pred[1, :]

plt.figure(figsize=(12, 4))
plt.plot(t[:-1], x_true, label="x real")
plt.plot(t[:-1], x_pred, "--", label="x predicho por EDMD/Koopman")
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Predicción iterada con operador de Koopman aproximado")
plt.legend()
plt.grid(True)
plt.show()

## 10. Error de predicción

Calculamos un error RMS para tener una medida numérica.

In [ ]:
rms_error = np.sqrt(np.mean((x_true - x_pred)**2))
peak = np.max(np.abs(x_true))

print("Error RMS:", rms_error)
print("Pico de |x|:", peak)
print("Error RMS relativo:", rms_error / peak)

## 11. Eigenvalores del operador discreto aproximado

Si $\mu_j$ es un eigenvalor de la matriz discreta $K$, entonces se relaciona con un exponente continuo aproximado mediante:

$$
\lambda_j \approx \frac{\log(\mu_j)}{\Delta t}.
$$

La parte imaginaria de $\lambda_j$ contiene frecuencias angulares.

In [ ]:
eigvals = np.linalg.eigvals(K)

plt.figure(figsize=(6, 6))
plt.scatter(eigvals.real, eigvals.imag)

circle = plt.Circle((0, 0), 1, fill=False)
plt.gca().add_artist(circle)

plt.axis("equal")
plt.xlabel("Parte real")
plt.ylabel("Parte imaginaria")
plt.title("Eigenvalores de la matriz discreta K")
plt.grid(True)
plt.show()

## 12. Frecuencias aproximadas

Para cada eigenvalor discreto $\mu_j$:

$$
\lambda_j = \frac{\log(\mu_j)}{\Delta t}.
$$

Si:

$$
\lambda_j = \sigma_j + j\omega_j,
$$

entonces la frecuencia en ciclos por unidad de tiempo es:

$$
f_j = \frac{\omega_j}{2\pi}.
$$

Como $\Omega=1$, la frecuencia fundamental esperada es:

$$
f_0 = \frac{1}{2\pi}\approx 0.1592.
$$

In [ ]:
lambda_cont = np.log(eigvals) / dt
freqs = lambda_cont.imag / (2*np.pi)

# Ordenamos por valor absoluto de frecuencia
order = np.argsort(np.abs(freqs))

print("Frecuencia fundamental esperada:", Omega/(2*np.pi))
print()
print("Primeras frecuencias aproximadas por Koopman/EDMD:")
for idx in order[:16]:
    print(f"mu = {eigvals[idx]: .4f}, lambda = {lambda_cont[idx]: .4f}, f = {freqs[idx]: .4f}")

## 13. Comparación con Fourier

Fourier analiza la señal ya producida $x(t)$.  
Koopman/EDMD intenta aproximar la evolución de observables del sistema.

Ambas lecturas se complementan.

In [ ]:
# FFT de la señal estacionaria
x_centered = x_true - np.mean(x_true)

freq_fft = np.fft.rfftfreq(len(x_centered), d=dt)
X_fft = np.abs(np.fft.rfft(x_centered))

plt.figure(figsize=(10, 4))
plt.plot(freq_fft, X_fft)
plt.xlim(0, 1.2)
plt.xlabel("Frecuencia [ciclos/unidad de tiempo]")
plt.ylabel("|FFT|")
plt.title("Espectro de Fourier de x(t)")
plt.grid(True)
plt.show()

print("Frecuencia fundamental esperada:", Omega/(2*np.pi))
print("Tercer armónico esperado:", 3*Omega/(2*np.pi))

## 14. Una predicción de corto plazo

La predicción iterada larga puede acumular error porque $K$ es una aproximación finita.  
Veamos una ventana más corta.

In [ ]:
window = int(20.0 / dt)

plt.figure(figsize=(12, 4))
plt.plot(t[:window], x_true[:window], label="x real")
plt.plot(t[:window], x_pred[:window], "--", label="x predicho por EDMD/Koopman")
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Predicción de corto plazo")
plt.legend()
plt.grid(True)
plt.show()

## 15. Experimentos sugeridos

Puedes modificar:

1. El diccionario de observables.
2. El paso de muestreo `dt`.
3. La fuerza de excitación `gamma`.
4. La no linealidad cúbica `beta`.
5. El tiempo eliminado como transitorio.

Preguntas útiles:

- ¿Qué pasa si eliminas los observables trigonométricos?
- ¿Qué pasa si agregas términos de cuarto o quinto grado?
- ¿Los eigenvalores se acercan mejor a los armónicos esperados?
- ¿La predicción de largo plazo mejora o empeora?

## 16. Lectura conceptual

Fourier pregunta:

$$
\text{¿Qué frecuencias tiene esta señal?}
$$

Koopman pregunta:

$$
\text{¿Qué observables evolucionan como modos exponenciales?}
$$

Es decir, buscamos funciones $\phi_j$ tales que:

$$
U^t \phi_j = e^{\lambda_j t}\phi_j.
$$

En una implementación finita como EDMD, no encontramos el Koopman exacto.  
Encontramos una aproximación proyectada sobre el diccionario elegido.

Por eso la frase clave es:

> La potencia de Koopman no está en volver lineal al sistema original, sino en revelar una estructura lineal aproximada en el espacio de observables.